# 04 · Retrieval evaluation: which method finds the right page?
Charts for LOGBOOK §7.2. **60 synthetic questions** (Opus wrote one per source passage). A hit = a retrieved chunk
overlaps the source page. Metrics @10: **Hit@k** (right page in top k?), **MRR** (1/rank of the first hit),
**nDCG** (rewards hits near the top).
⚠️ Synthetic questions share some wording with their source, so they **favour BM25** (P-34). Max's own questions are the reality check.

In [ ]:
import sys, json, pickle; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.max_colwidth', 70); pd.set_option('display.width', 200)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
ev = pd.read_csv('../data/eval_retrieval.csv').sort_values('mrr')
sw = pd.read_csv('../data/eval_sweeps.csv')
perq = pd.read_csv('../data/eval_retrieval_perq.csv', index_col=0)
ev[['method', 'hit1', 'hit5', 'mrr', 'ndcg', 'ms_per_query']].sort_values('mrr', ascending=False)

## 1. Quality per method

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5)); y = np.arange(len(ev)); h = .27
for k, (m, c) in enumerate([('hit5', 'tab:blue'), ('mrr', 'tab:orange'), ('ndcg', 'tab:green')]):
    ax.barh(y + (k - 1) * h, ev[m], h, label=m, color=c)
ax.set_yticks(y, ev.method); ax.set_xlim(.4, 1.02); ax.legend(loc='lower right'); ax.set_title('Retrieval quality (60 synthetic questions)')
plt.tight_layout(); plt.show()

**Reading:** the **embedding model** matters most (Qwen3 vs nomic: +15 pts Hit@5). **Euclid = cosine** (D1 answer).
Hybrid beats either side alone, and a **reranker** adds the last few points. Claude rerank is best (D19).

## 2. Quality vs. speed: what you pay for the last points

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(ev.ms_per_query, ev.mrr, s=40)
for x in ev.itertuples(): ax.annotate(x.method, (x.ms_per_query, x.mrr), fontsize=7, xytext=(4, 2), textcoords='offset points')
ax.set_xscale('log'); ax.set(xlabel='ms per query (log)', ylabel='MRR', title='Rerankers: ~100× slower for +0.04–0.11 MRR')
plt.tight_layout(); plt.show()

## 3. Sweeps: how much of each signal to mix
α = weight of the first signal in a weighted (min-max normalized) fusion.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for a, (name, g) in zip(ax, sw.groupby('sweep')):
    a.plot(g.alpha, g.hit5, 'o-', label='Hit@5'); a.plot(g.alpha, g.mrr, 's-', label='MRR'); a.set(title=name, xlabel='α')
ax[0].legend(); plt.tight_layout(); plt.show()

**Reading:** cos↔Euclid is **flat** (length adds nothing). Dense↔BM25 peaks at α≈0.75: **both together beat either alone**.

## 4. Per question: which questions does everyone miss? (Hit@5)
Rows = questions, columns = methods. A row that's red for every method usually means a **bad gold label** or a
question that is genuinely hard. These are the rows worth checking in Max's spot-check.

In [ ]:
order = perq.mean(axis=1).sort_values().index
fig, ax = plt.subplots(figsize=(11, 9)); ax.imshow(perq.loc[order].T, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_yticks(range(perq.shape[1]), perq.columns, fontsize=8); ax.set_xlabel('question (sorted: hardest left)')
ax.set_title('Hit@5 per question and method (green = found)'); plt.tight_layout(); plt.show()
tests = pd.read_json('../testset/synthetic.jsonl', lines=True)
hard = perq.mean(axis=1).sort_values().head(5)
pd.DataFrame({'question': tests.question[hard.index], 'share of methods that hit': hard.round(2)})